# Scatter Plot - Data Analysis Notebook

In [1]:
import pandas as pd
import json
import matplotlib.pyplot as plt

In [2]:
# Read json files from ../src/json/fips.json
# FIPS 10-4 to Country Name mapping
with open('../src/json/fips.json', 'r') as f:
    fips = json.load(f)

In [3]:
# Function to convert 2-letter FIPS 10-4 country codes to country names
def code_to_country(code):
    try:
        country = fips[code]
        return country
    except:
        raise ValueError(f"Invalid country code: {code}")

In [4]:
# Load the mentions dataset from data/GDELT/weekly_media_attention_query_class34_better_disambiguation.csv
# The old dataset
# path_mentions = "../data/GDELT/weekly_media_attention_query_class34_better_disambiguation.csv"

# The new one
# path_mentions = "../data/GDELT/2026_weekly_media_attention_query_class34_better_disambiguation.csv"
# df_mentions = pd.read_csv(path_mentions)
# df_mentions.head()

gdelt_path = "../data/GDELT/"
df_keys = pd.read_parquet(gdelt_path + "gdelt_keys.parquet")
df_keys.head()

,conflict_id,mention_week,conflict_country,actor1_country,actor2_country,media_country
0,0,2026-01-12,UK,UNK,GBR,UP
1,1,2026-01-12,FR,FRA,USA,US
2,2,2026-01-12,IS,GBR,UNK,UK
3,3,2026-01-12,UP,EUR,RUS,UK
4,4,2026-01-12,EZ,SVK,UNK,EZ


In [5]:
df_volume = pd.read_parquet(gdelt_path + "gdelt_volume_metrics.parquet")
df_volume.head()

,conflict_id,mentions_count,distinct_events,distinct_media_sources
0,0,16,13,11
1,1,101,28,71
2,2,8,8,5
3,3,208,5,105
4,4,6,5,4


In [6]:
df_mention_types = pd.read_parquet(gdelt_path + "gdelt_conflict_types.parquet")
df_mention_types.head()

,conflict_id,verbal_conflict_mentions,material_conflict_mentions,verbal_conflict_unique_events,material_conflict_unique_events
0,0,3,13,3,10
1,1,41,60,11,17
2,2,6,2,6,2
3,3,207,1,4,1
4,4,5,1,4,1


In [7]:
# join just df_keys and df_mention_types on conflict_id
df_mentions = pd.merge(df_keys, df_mention_types, on='conflict_id')
df_mentions.head()

,conflict_id,mention_week,conflict_country,actor1_country,actor2_country,media_country,verbal_conflict_mentions,material_conflict_mentions,verbal_conflict_unique_events,material_conflict_unique_events
0,0,2026-01-12,UK,UNK,GBR,UP,3,13,3,10
1,1,2026-01-12,FR,FRA,USA,US,41,60,11,17
2,2,2026-01-12,IS,GBR,UNK,UK,6,2,6,2
3,3,2026-01-12,UP,EUR,RUS,UK,207,1,4,1
4,4,2026-01-12,EZ,SVK,UNK,EZ,5,1,4,1


In [8]:
df_mentions = pd.merge(df_mentions, df_volume, on='conflict_id')
df_mentions.head()

,conflict_id,mention_week,conflict_country,actor1_country,actor2_country,media_country,verbal_conflict_mentions,material_conflict_mentions,verbal_conflict_unique_events,material_conflict_unique_events,mentions_count,distinct_events,distinct_media_sources
0,0,2026-01-12,UK,UNK,GBR,UP,3,13,3,10,16,13,11
1,1,2026-01-12,FR,FRA,USA,US,41,60,11,17,101,28,71
2,2,2026-01-12,IS,GBR,UNK,UK,6,2,6,2,8,8,5
3,3,2026-01-12,UP,EUR,RUS,UK,207,1,4,1,208,5,105
4,4,2026-01-12,EZ,SVK,UNK,EZ,5,1,4,1,6,5,4


In [9]:
# Print unique values in the 'mention_week' column sorted by most recent date
print(sorted(df_mentions['mention_week'].unique(), reverse=True))

['2026-01-12', '2026-01-05', '2025-12-29', '2025-12-22', '2025-12-15', '2025-12-08', '2025-12-01', '2025-11-24', '2025-11-17', '2025-11-10', '2025-11-03', '2025-10-27', '2025-10-20', '2025-10-13', '2025-10-06', '2025-09-29', '2025-09-22', '2025-09-15', '2025-09-08', '2025-09-01', '2025-08-25', '2025-08-18', '2025-08-11', '2025-08-04', '2025-07-28', '2025-07-21', '2025-07-14', '2025-07-07', '2025-06-30', '2025-06-09', '2025-06-02', '2025-05-26', '2025-05-19', '2025-05-12', '2025-05-05', '2025-04-28', '2025-04-21', '2025-04-14', '2025-04-07', '2025-03-31', '2025-03-24', '2025-03-17', '2025-03-10', '2025-03-03', '2025-02-24', '2025-02-17', '2025-02-10', '2025-02-03', '2025-01-27', '2025-01-20', '2025-01-13', '2025-01-06', '2024-12-30', '2024-12-23', '2024-12-16', '2024-12-09', '2024-12-02', '2024-11-25', '2024-11-18', '2024-11-11', '2024-11-04', '2024-10-28', '2024-10-21', '2024-10-14', '2024-10-07', '2024-09-30', '2024-09-23', '2024-09-16', '2024-09-09', '2024-09-02', '2024-08-26', '2024

In [10]:
# Since we are in 2026 now and we want to analyze 2025 data,
# we only keep GDELT data up to the 2025-12-31 week
df_mentions = df_mentions[df_mentions['mention_week'] <= '2025-12-31']

In [11]:
# Load the fatalities dataset from data/ACLED/number_of_reported_fatalities_by_country-year_as-of-09Jan2026.CSV
path_fatalities = "../data/ACLED/number_of_reported_fatalities_by_country-year_as-of-09Jan2026.CSV"
df_fatalities = pd.read_csv(path_fatalities, encoding='utf-8', sep=';')

# Display the first few rows of the dataframe
df_fatalities.head()

,COUNTRY,YEAR,FATALITIES
0,Afghanistan,2017,36360
1,Afghanistan,2018,42991
2,Afghanistan,2019,41419
3,Afghanistan,2020,30977
4,Afghanistan,2021,42425


In [12]:
# Removing the rows with YEAR == 2026
df_fatalities = df_fatalities[df_fatalities['YEAR'] != 2026]
df_fatalities.head(10)

,COUNTRY,YEAR,FATALITIES
0,Afghanistan,2017,36360
1,Afghanistan,2018,42991
2,Afghanistan,2019,41419
3,Afghanistan,2020,30977
4,Afghanistan,2021,42425
5,Afghanistan,2022,4152
6,Afghanistan,2023,1152
7,Afghanistan,2024,1362
8,Afghanistan,2025,915
10,Akrotiri and Dhekelia,2018,0


Create a new dataset with country name, number of mentions, and number of fatalities for the scatter plot visualization.

The year will be also kept to allow visualizing the evolution over time.

Pay attention: here **we are considering all the conflict mentions**, not only material conflict ones.

In [13]:
# First, add column YEAR (column 'mention_week' contains the year at the start)
df_mentions_year = df_mentions.copy()
df_mentions_year['year'] = df_mentions_year['mention_week'].str[:4].astype(int)
# Remove entries with country codes not in fips keys
df_mentions_year = df_mentions_year[df_mentions_year['conflict_country'].isin(set(fips.keys()))]
# Aggregate the number of mentions by country and year
mentions_agg = df_mentions_year.groupby(['conflict_country', 'year'])['mentions_count'].sum().reset_index()
# Rename columns for clarity
mentions_agg.columns = ['COUNTRY', 'YEAR', 'MENTIONS']
# Map country codes to country names
mentions_agg['COUNTRY'] = mentions_agg['COUNTRY'].apply(code_to_country)
# If a country appears multiple times, sum the mentions
mentions_agg = mentions_agg.groupby(['COUNTRY', 'YEAR'])['MENTIONS'].sum().reset_index()

# Select relevant columns
fatalities_agg = df_fatalities[['COUNTRY', 'YEAR', 'FATALITIES']]
# Merge the two datasets on country name
merged_df = pd.merge(mentions_agg, fatalities_agg, on=['COUNTRY', 'YEAR'], how='inner')
# Display the merged dataframe
merged_df.head()

,COUNTRY,YEAR,MENTIONS,FATALITIES
0,Afghanistan,2017,989643,36360
1,Afghanistan,2018,902235,42991
2,Afghanistan,2019,553219,41419
3,Afghanistan,2020,368395,30977
4,Afghanistan,2021,914148,42425


In [14]:
# Show highest values of mentions and fatalities
print("Top 10 countries by mentions in 2025:")
print(merged_df.sort_values(by='MENTIONS', ascending=False).head(10))
print()

print("Top 10 countries by fatalities in 2025:")
print(merged_df.sort_values(by='FATALITIES', ascending=False).head(10))
print()

print("Bottom 10 countries by mentions in 2025:")
print(merged_df.sort_values(by='MENTIONS', ascending=True).head(10))
print()

print("Bottom 10 countries by fatalities in 2025:")
print(merged_df.sort_values(by='FATALITIES', ascending=True).head(10))

Top 10 countries by mentions in 2025:
            COUNTRY  YEAR  MENTIONS  FATALITIES
1684  United States  2020  16123908          74
1685  United States  2021  13000763          87
1687  United States  2023  12237312          66
1688  United States  2024  11765870          33
1689  United States  2025  10965487          54
1686  United States  2022   9646538          98
778          Israel  2024   6790269          99
777          Israel  2023   5567365        1772
1331         Russia  2022   5016223          97
772          Israel  2018   4263326           7

Top 10 countries by fatalities in 2025:
          COUNTRY  YEAR  MENTIONS  FATALITIES
1668      Ukraine  2025   1792913       78347
1667      Ukraine  2024   2030567       73570
750          Iraq  2016   1890947       56032
1549        Syria  2017   2713455       54122
1     Afghanistan  2018    902235       42991
4     Afghanistan  2021    914148       42425
2     Afghanistan  2019    553219       41419
1665      Ukraine  2022  

In [15]:
# Save the merged dataset to a new CSV file
output_path = "../data/processed/scatter_plot.csv"
merged_df.to_csv(output_path, index=False)
print(f"Merged dataset saved to {output_path}")

Merged dataset saved to ../data/processed/scatter_plot.csv


### Taking only the material conflict mentions and not the verbal conflict ones

In [16]:
# Create a new dataset with country name, number of mentions in 2025, and number of fatalities in 2025

df_mentions_year = df_mentions.copy()
df_mentions_year['year'] = df_mentions_year['mention_week'].str[:4].astype(int)
df_mentions_year = df_mentions_year[df_mentions_year['conflict_country'].isin(set(fips.keys()))]
mentions_agg = df_mentions_year.groupby(['conflict_country', 'year'])['material_conflict_mentions'].sum().reset_index()
mentions_agg.columns = ['COUNTRY', 'YEAR', 'MENTIONS']
mentions_agg['COUNTRY'] = mentions_agg['COUNTRY'].apply(code_to_country)
mentions_agg = mentions_agg.groupby(['COUNTRY', 'YEAR'])['MENTIONS'].sum().reset_index()

# Select relevant columns
fatalities_agg = df_fatalities[['COUNTRY', 'YEAR', 'FATALITIES']]
# Merge the two datasets on country name
merged_df = pd.merge(mentions_agg, fatalities_agg, on=['COUNTRY', 'YEAR'], how='inner')
# Display the merged dataframe
merged_df.head()

,COUNTRY,YEAR,MENTIONS,FATALITIES
0,Afghanistan,2017,713597,36360
1,Afghanistan,2018,680361,42991
2,Afghanistan,2019,376789,41419
3,Afghanistan,2020,246811,30977
4,Afghanistan,2021,563730,42425


In [17]:
# Save the merged dataset to a new CSV file
output_path = "../data/processed/scatter_plot_material_conflict.csv"
merged_df.to_csv(output_path, index=False)
print(f"Merged dataset saved to {output_path}")

Merged dataset saved to ../data/processed/scatter_plot_material_conflict.csv
